[← Back to the Capstone](<../capstone-online-store-analysis.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# Capstone - The Online Store: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/capstone-online-store-analysis-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

Every answer here runs on SQL Server, so run both setup cells first. In Colab the second one takes a few minutes the first time; on your own machine it only checks the connection.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


Exercises 2 to 4 use the `#customer_first_orders` table from section 7 of the capstone, so this cell rebuilds it:

In [3]:
%%tsql
DROP TABLE IF EXISTS #customer_first_orders;

CREATE TABLE #customer_first_orders (
    CustomerID INT NOT NULL PRIMARY KEY,
    first_order_id INT NOT NULL,
    first_order_date DATE NOT NULL,
    first_order_had_bike BIT NOT NULL,
    second_order_date DATE NULL,
    CHECK (second_order_date >= first_order_date)
);

WITH online_orders AS (
    SELECT
        SalesOrderID,
        CustomerID,
        OrderDate,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY
                OrderDate,
                SalesOrderID
        ) AS order_number
    FROM Sales.SalesOrderHeader
    WHERE OnlineOrderFlag = 1
),
bike_orders AS (
    SELECT DISTINCT
        d.SalesOrderID
    FROM Sales.SalesOrderDetail AS d
    INNER JOIN Production.Product AS p
        ON p.ProductID = d.ProductID
    INNER JOIN Production.ProductSubcategory AS s
        ON s.ProductSubcategoryID = p.ProductSubcategoryID
    WHERE s.ProductCategoryID = 1
)
INSERT INTO #customer_first_orders (
    CustomerID,
    first_order_id,
    first_order_date,
    first_order_had_bike,
    second_order_date
)
SELECT
    f.CustomerID,
    f.SalesOrderID,
    f.OrderDate,
    CASE WHEN b.SalesOrderID IS NULL THEN 0 ELSE 1 END,
    s.OrderDate
FROM online_orders AS f
LEFT JOIN bike_orders AS b
    ON b.SalesOrderID = f.SalesOrderID
LEFT JOIN online_orders AS s
    ON s.CustomerID = f.CustomerID
   AND s.order_number = 2
WHERE f.order_number = 1;

18484 rows inserted


## 1. The earlier drop

In [4]:
%%tsql
SELECT
    m.Name AS bike_model,
    MIN(h.OrderDate) AS first_sold_online,
    MAX(h.OrderDate) AS last_sold_online,
    CAST(AVG(d.UnitPrice) AS DECIMAL(10, 0)) AS avg_unit_price
FROM Sales.SalesOrderHeader AS h
INNER JOIN Sales.SalesOrderDetail AS d
    ON d.SalesOrderID = h.SalesOrderID
INNER JOIN Production.Product AS p
    ON p.ProductID = d.ProductID
INNER JOIN Production.ProductModel AS m
    ON m.ProductModelID = p.ProductModelID
INNER JOIN Production.ProductSubcategory AS s
    ON s.ProductSubcategoryID = p.ProductSubcategoryID
WHERE h.OnlineOrderFlag = 1
  AND s.ProductCategoryID = 1
GROUP BY
    m.Name
ORDER BY
    first_sold_online,
    bike_model;

bike_model,first_sold_online,last_sold_online,avg_unit_price
Mountain-100,2011-05-31 00:00:00,2012-05-29 00:00:00,3387
Road-150,2011-05-31 00:00:00,2012-05-29 00:00:00,3578
Road-650,2011-05-31 00:00:00,2013-05-29 00:00:00,757
Mountain-200,2012-05-30 00:00:00,2014-05-30 00:00:00,2232
Road-250,2012-05-30 00:00:00,2014-05-30 00:00:00,2339
Road-550-W,2012-05-30 00:00:00,2014-05-30 00:00:00,1090
Road-350-W,2013-05-30 00:00:00,2014-05-29 00:00:00,1701
Road-750,2013-05-30 00:00:00,2014-05-30 00:00:00,540
Touring-1000,2013-05-30 00:00:00,2014-05-30 00:00:00,2384
Touring-2000,2013-05-30 00:00:00,2014-05-30 00:00:00,1215


The bike range changed on 30 May 2012. The two most expensive models, Mountain-100 and Road-150, averaging around $3,400 to $3,600, were last sold online on 29 May 2012, and the next day Mountain-200, Road-250, and Road-550-W replaced them, averaging roughly $1,100 to $2,300. The average order fell because the bikes got cheaper, not because people bought less. The table also shows the range changing again on 30 May 2013, when seven more models arrived, most of them cheaper still, and it shows every model's last online sale falling in May 2014, which fits section 5's suspicion about June 2014. Every change in this dataset lands within a few days of 30 May, a sign of how the sample data was generated, and the sort of pattern worth noticing in any dataset before treating dates as real events.

## 2. How much does the window matter?

In [5]:
%%tsql
SELECT
    CASE WHEN first_order_had_bike = 1 THEN 'Bike' ELSE 'No bike' END AS first_order,
    COUNT(*) AS customers,
    CAST(100.0 * SUM(CASE WHEN second_order_date <= DATEADD(day, 30, first_order_date) THEN 1 ELSE 0 END)
         / COUNT(*) AS DECIMAL(5, 1)) AS pct_within_30_days,
    CAST(100.0 * SUM(CASE WHEN second_order_date <= DATEADD(day, 90, first_order_date) THEN 1 ELSE 0 END)
         / COUNT(*) AS DECIMAL(5, 1)) AS pct_within_90_days,
    CAST(100.0 * SUM(CASE WHEN second_order_date <= DATEADD(day, 180, first_order_date) THEN 1 ELSE 0 END)
         / COUNT(*) AS DECIMAL(5, 1)) AS pct_within_180_days
FROM #customer_first_orders
WHERE first_order_date >= '2013-06-01'
  AND first_order_date < '2013-12-01'
GROUP BY
    first_order_had_bike
ORDER BY
    first_order;

first_order,customers,pct_within_30_days,pct_within_90_days,pct_within_180_days
Bike,1518,0.0,0.7,3.4
No bike,4456,4.7,12.1,20.4


At 30 days, no bike buyer had come back and 4.7 percent of the others had; at 90 days, 0.7 against 12.1 percent; at 180 days, 3.4 against 20.4. The gap holds at every window, so the finding doesn't depend on picking 180, which is exactly what this check is for: if the conclusion had flipped between windows, the write-up would have had to say the answer depends on the definition.

Keeping the same group of customers for every window matters: a 30-day window would let you include customers who first ordered as late as April 2014, but then each column would describe a different group of people, and the comparison across columns would mean nothing.

## 3. How long until they come back?

In [6]:
%%tsql
WITH came_back AS (
    SELECT
        first_order_had_bike,
        DATEDIFF(day, first_order_date, second_order_date) AS days_to_return
    FROM #customer_first_orders
    WHERE first_order_date >= '2013-06-01'
      AND first_order_date < '2013-12-01'
      AND second_order_date <= DATEADD(day, 180, first_order_date)
)
SELECT DISTINCT
    CASE WHEN first_order_had_bike = 1 THEN 'Bike' ELSE 'No bike' END AS first_order,
    COUNT(*) OVER (PARTITION BY first_order_had_bike) AS customers,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_to_return)
        OVER (PARTITION BY first_order_had_bike) AS median_days_to_return
FROM came_back
ORDER BY
    first_order;

first_order,customers,median_days_to_return
Bike,52,116.5
No bike,910,74.0


The no-bike customers who came back did so after a median of 74 days, the bike customers after 116.5. Only 52 bike customers came back at all, though, so their median rests on a small group and would move noticeably with a few more customers; a write-up should give the group size next to the number.

On SQL Server, `PERCENTILE_CONT` is only a window function, so it returns the median on every row of its partition, and `DISTINCT` collapses those identical rows to one per group. It's an awkward shape, and a common interview follow-up.

## 4. Stretch: what do they buy first?

In [7]:
%%tsql
WITH first_orders AS (
    SELECT
        first_order_id AS SalesOrderID
    FROM #customer_first_orders
    WHERE first_order_had_bike = 0
      AND first_order_date >= '2013-06-01'
      AND first_order_date < '2013-12-01'
)
SELECT TOP 5
    s.Name AS subcategory,
    COUNT(DISTINCT o.SalesOrderID) AS first_orders
FROM first_orders AS o
INNER JOIN Sales.SalesOrderDetail AS d
    ON d.SalesOrderID = o.SalesOrderID
INNER JOIN Production.Product AS p
    ON p.ProductID = d.ProductID
INNER JOIN Production.ProductSubcategory AS s
    ON s.ProductSubcategoryID = p.ProductSubcategoryID
GROUP BY
    s.Name
ORDER BY
    first_orders DESC;

subcategory,first_orders
Tires and Tubes,2774
Helmets,1165
Bottles and Cages,833
Jerseys,605
Fenders,459


Tires and tubes come first by a long way, in 2,774 of the 4,456 first orders without a bike, followed by helmets, bottles and cages, jerseys, and fenders. These are mostly things a cyclist who already owns a bike needs, which fits the reading in section 7: many no-bike first orders look like existing riders buying parts and kit, people with a reason to come back for more. It's a plausible story rather than a proven one, since the data can't say whether these customers own a bike from somewhere else.

---

[← Back to the Capstone](<../capstone-online-store-analysis.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).